# DEEPX Tutorial 02 - Usage of DX-APP

In this second tutorial, we will introduce DX-APP and learn how to run a compiled DXNN model in an AI application.

## What is DX-APP?

**DX-APP** is a sample application that demonstrates how to run compiled models on DEEPX NPUs through DX-RT. It includes ready-to-use code for common vision tasks such as object detection, face
recognition, and image classification. DX-APP helps developers quickly set up the runtime environment
and serves as a template for building and customizing their own AI applications.

For more details, find the overview from [dx_app git repo](https://github.com/DEEPX-AI/dx_app/blob/main/README.md) or download DX-APP User Guide from 👉 [here](https://developer.deepx.ai/download/?id=580)!

> Note: To download the User Guide, you must log in to https://developer.deepx.ai/ first.

Let's see the file structure of DX-APP:

In [ ]:
# Load all SDK paths from dx-tutorials/config.json.
import os
root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")
%run "$root_path/tutorial_paths.py"
print_tutorial_paths()
%cd $DX_ALL_SUITE_DIR

In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** demos are optimized to showcase pre-compiled models on DEEPX NPUs with minimal setup.
Each demo represents a common AI task and can be executed using images, videos, or live camera
input.

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP Repository Layout:**
```shell
dx_app/
├── src/
│   ├── cpp_example/            # C++ end-to-end examples (348 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Model-specific factories, configs, and entry points
│   │   └── common/             # Shared C++ runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   42 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   16 task-specific sync/async runner pairs
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   13 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader
│   │       ├── trackers/       #   Shared IoU tracker
│   │       ├── third_party/    #   Vendored header dependencies
│   │       └── utility/        #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/         # Python end-to-end examples (348 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Four execution/postprocessing variants per model
│   │   └── common/             # Shared Python runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   41 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   SyncRunner, AsyncRunner, arguments, and run utilities
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   14 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader and schema
│   │       ├── trackers/       #   Shared IoU tracker
│   │       └── utility/        #   Labels, preprocessing, profiling, and geometry helpers
│   ├── postprocess/            # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           └── dx_postprocess/ # pybind11 bindings wrapping src/postprocess/
├── config/
│   ├── model_registry.json     # Model registry — single source of truth
│   ├── test_models.conf        # Test model configuration
│   └── README.md               # Config directory documentation
├── scripts/                    # Developer tools, validation, and helper scripts
├── tests/                      # C++, Python, script, helper, and Windows tests
├── assets/                     # Links to downloaded models and videos
├── sample/                     # Bundled images and task-specific sample data
├── bin/                        # C++ executables generated by the build
├── cmake/                      # x86_64 and aarch64 toolchain files
├── extern/                     # External header dependencies
├── build.sh / build.bat        # Linux and Windows build scripts
├── setup.sh / setup.bat        # Model, video, and dependency setup
├── setup_sample_models.sh      # Download the sample model set only
├── setup_sample_videos.sh      # Download the sample video set only
├── run_demo.sh / run_demo.bat  # Interactive demo launcher
├── run_tc.sh                   # Unified test runner for example tests
├── install.sh                  # Dependency and OpenCV installer
└── docs/                       # Detailed documentation
```
---
**DX-APP provides both C++ and Python examples:**

Both `cpp_example` and `python_example` follow the same task-first structure.

Both languages currently provide the same 22 task directories:

- 3d_object_detection/
- attribute_recognition/
- classification/
- depth_estimation/
- embedding/
- face_alignment/
- face_detection/
- hand_detection/
- hand_landmark/
- image_denoising/
- image_enhancement/
- instance_segmentation/
- keypoint_detection/
- obb_detection/
- object_detection/
- object_pose_estimation/
- panoptic_driving_perception/
- pose_estimation/
- ppu/
- reid/
- semantic_segmentation/
- super_resolution/

## 1. Prerequisites

1. Move to `dx_app` directory:

In [ ]:
# Move to the configured DX-APP directory.
%cd $DX_APP_DIR

2. Download the required models and sample videos with `setup.sh`.

`setup.sh` is interactive, so run it in a separate terminal instead of a Notebook code cell.

3. Select **File > New > Terminal**. A new terminal starts in `dx-tutorials`. Run the next code cell, then copy and run the displayed commands in that terminal.

In [ ]:
import shlex
from IPython.display import Markdown, display

dx_app_terminal_path = shlex.quote(str(DX_APP_DIR))
display(Markdown(
    "### Run in a separate terminal\n\n"
    "![Open a terminal](assets/open-terminal.png)\n\n"
    "Copy and run these commands:\n\n"
    f"```bash\ncd {dx_app_terminal_path}\n./setup.sh --no-force\n```"
))

> ![](assets/sc-setup.png)

4. If you have at least 29 GB of free storage, press **Enter** to download all Model Zoo models.

5. Otherwise, download only the models for the following tutorials using the code cell below:

In [ ]:
# Download *.dxnn models for quick validation

SELECTED_MODELS = shlex.join([
    "resnet50",
    "yolov7",
    "yolov7_ppu",
    "yolov8s_pose",
    "yolov8n_seg"
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat $MODEL_MANIFEST_PATH | grep name

# Download selected models only
!cd $DX_APP_DIR && bash setup.sh --models $SELECTED_MODELS --no-force

6. Verify that both models and videos are downloaded as expected:

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd $DX_APP_DIR && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd $DX_APP_DIR && ls assets/models | wc -l

# Downloaded model list
!cd $DX_APP_DIR && tree assets/models

In [ ]:
# video files for demo inputs
!cd $DX_APP_DIR && tree assets/videos

## 2. Run Demos

**DX-APP** is a set of ready-made demo applications that show how to run compiled models on DEEPX NPUs, including classification, detection, segmentation, pose estimation, and many others.

Users can learn and build applications by exploring the diverse DX-APP demos.

### 2.1. Classification
- **bin/resnet50_async** is the DX-APP executable that runs a pre-trained image classification model and outputs the top predicted class.

In [ ]:
%cd $DX_APP_DIR 

#!ls bin
!./bin/resnet50_async -h
#!./bin/resnet50_async -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg

### 2.2. Object Detection
- **bin/yolov\<version>_async** is the DX_APP demo binary that runs YOLO-based object detection models to detect objects in images, video or camera.

In [ ]:
%cd $DX_APP_DIR 

# Find other yolo based reference excutable files
!ls ./bin/yolo*

#### Execution Modes

DX-APP provides different execution modes to suit different application requirements:

| Mode | Main characteristic | Recommended use |
|---|---|---|
| Synchronous | Processes each frame sequentially | Simple flow and latency inspection |
| Asynchronous | Overlaps pipeline stages | Higher throughput |
| PPU | Offloads supported post-processing work | Lower host CPU overhead |

- **Synchronous application**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">


In [ ]:
%cd $DX_APP_DIR 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_sync -h

# Image input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **Asynchronous application**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">



In [ ]:
%cd $DX_APP_DIR 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
#!./bin/yolov7_async -h

# Image input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (This IP address only works in Korea)
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- PPU vs non-PPU

In [ ]:
%cd $DX_APP_DIR

# PPU disabled
!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/dron-citry-road2.mov

In [ ]:
%cd $DX_APP_DIR

# PPU enabled
!./bin/yolov7_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/dron-citry-road2.mov

### 2.3. Pose Estimation
- **bin/yolov8s_pose_sync** is the DX_APP demo binary that runs a pose estimation model to detect people and estimate keypoints in images, video or camera.

In [ ]:
%cd $DX_APP_DIR

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8s_pose_sync -h

# Image input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov

# Camera input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4. Segmentation
- **bin/yolov8n_seg_async** is the DX_APP demo binary that runs an instance segmentation model to produce pixel-wise class labels for an image, video or camera.

In [ ]:
%cd $DX_APP_DIR

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8n_seg_async -h

# Image input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4

# Camera input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5. Demo Script

`run_demo.sh` is an interactive launcher for both C++ and Python demos.

Select **File > New > Terminal**. A new terminal starts in `dx-tutorials`. Run the next code cell, then copy and run the displayed commands in that terminal.

In [ ]:
import shlex
from IPython.display import Markdown, display

dx_app_terminal_path = shlex.quote(str(DX_APP_DIR))
display(Markdown(
    "### Run in a separate terminal\n\n"
    "![Open a terminal](assets/open-terminal.png)\n\n"
    "Copy and run these commands:\n\n"
    f"```bash\ncd {dx_app_terminal_path}\n./run_demo.sh\n```"
))

> ![](assets/sc-run_demo.png)

The launcher provides 23 ready-to-run demos across the DX-APP task categories.

If you select `0` Object Detection, the following execution options are available:
> ![](assets/sc-run_demo2.png)

Options [1, 2] run C++-based applications, and options [3–6] run Python-based applications.

> Note: Activate the Python virtual environment where `dx-engine` is installed to run Python-based applications.

Example:
> ![](assets/sc-run_demo3.png)


### 2.6. Extract sample applications written in C++ and Python

`scripts/dx_tool.sh` is an interactive terminal tool for exporting standalone examples written in both C++ and Python.

1. Run the following code cell and Select **File > New > Terminal**.
2. A new terminal starts in `dx-tutorials`.
3. Copy the following commands generated after running the code cell.
4. Then, `dx-tool.sh` will work and you can see the extract menu.

In [ ]:
# Run here!!
import shlex
from IPython.display import Markdown, display

dx_app_terminal_path = shlex.quote(str(DX_APP_DIR))
display(Markdown(
    "### Run in a separate terminal\n\n"
    "![Open a terminal](assets/open-terminal.png)\n\n"
    "Copy and run these commands:\n\n"
    f"```bash\ncd {dx_app_terminal_path}\n./scripts/dx-tool.sh\n```"
))

If the `dx-tool.sh` runs, you can see the following menus. 
- Choose `3` to clarify the available model list you are interested in.
- Choose `2` to extract the sample code.
- Select the programming language. (C++, Python)
- Type the category and model for your application. (e.g., `object_detection/yolo26s`)

> ![](assets/sc-extract.png)

A few seconds later, sample code will be generated to `output` directory.

> ![](assets/sc-extract2.png)

## 3. Summary

### 3.1. DX-APP workflow completed

```text
Load SDK paths
       │
       ▼
Prepare selected models and videos
       │
       ▼
Run C++ or Python examples
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Use run_demo.sh or export a standalone example
```

### 3.2. Learning dashboard

| Area | What you can now do |
|---|---|
| SDK paths | Load `DX_APP_DIR` and related paths from `config.json` |
| Model setup | Download only the models and videos required for an exercise |
| Example selection | Find examples by task, model, language, and execution mode |
| Execution modes | Choose synchronous, asynchronous, or PPU execution for the intended goal |
| Demo launcher | Start an installed example with `run_demo.sh` |
| Standalone code | Export a focused C++ or Python example with `scripts/dx_tool.sh` |

### 3.3. Completion checklist

- [ ] Confirmed that the DEEPX NPU is visible to DX-RT
- [ ] Prepared the required DX-APP model and video assets
- [ ] Ran at least one end-to-end inference example
- [ ] Compared the available execution modes
- [ ] Located a reference implementation for a target task
- [ ] Opened `run_demo.sh` or exported a standalone example

> **Next:** Continue to Tutorial 03 to adapt a trained model, compile it to DXNN, and integrate it with a DX-APP runtime configuration.
